In [4]:
from demoparser2 import DemoParser
import pandas as pd
from pathlib import Path
import numpy as np

In [5]:
#BASE_DIR = Path(__file__).resolve().parent
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'


In [6]:

PLAYER_PROPS = [
    "team_name",
    "tick",
    "total_rounds_played",
    "player_name",
    "team_num",
    "health",
    "X", "Y", "Z",      # Position
    "yaw",              # Looking direction
    "is_alive",
    "inventory",         # To see what they are carrying
]

EVENT_PROPS = [
    'round_announce_match_start',
    'cs_round_final_beep',
    'round_freeze_end',
    'bomb_planted',
    'bomb_exploded',
    'bomb_defused',
    'cs_win_panel_match',
    'announce_phase_end',
    'player_team'
]



In [72]:
def get_data(path):
    parser = DemoParser(path)
    df = parser.parse_ticks(PLAYER_PROPS)

    return df

In [7]:
files = [str(p) for p in list(DATA_DIR.rglob('*.dem'))]

In [74]:
df = get_data(files[0])

In [ ]:
parser.parse_header()

#Need to get map out of map_name and demo_version_guid or a running guide can be demo_id

In [45]:
parser = DemoParser(files[0])
test = parser.parse_events(EVENT_PROPS)

#We will need to get bomb_planted and make add a bomb_planted marker for the time

In [46]:
test

[('bomb_exploded',
     site    tick user_name       user_steamid
  0   263   20619    jcobbb  76561198178737429
  1   263   30014  karrigan  76561197989430253
  2   326   67507  karrigan  76561197989430253
  3   326   76598     broky  76561198201620490
  4   263  137272     ZywOo  76561198113666193
  5   326  159807     mezii  76561197973140692),
 ('player_team',
      disconnect  isbot  oldteam  silent  team    tick user_name  \
  0        False  False        2    True     3  104993  karrigan   
  1        False  False        3    True     2  104993     ZywOo   
  2        False  False        3    True     2  104993      apEX   
  3        False  False        3    True     2  104993      ropz   
  4        False  False        3    True     2  104993    flameZ   
  5        False  False        2    True     3  104993    jcobbb   
  6        False  False        3    True     2  104993     mezii   
  7        False  False        2    True     3  104993    frozen   
  8        False  Fal

In [75]:
df

,inventory,total_rounds_played,player_name,health,team_num,team_name,X,yaw,Y,Z,is_alive,tick,steamid,name
0,"[Karambit, Glock-18]",0,karrigan,100.0,2.0,TERRORIST,-1947.010010,-47.999954,-965.109985,-415.968750,True,0,76561197989430253,karrigan
1,"[Butterfly Knife, USP-S]",0,ZywOo,100.0,3.0,CT,2552.000000,-152.499847,-424.000000,-351.968750,True,0,76561198113666193,ZywOo
2,"[M9 Bayonet, USP-S]",0,apEX,100.0,3.0,CT,2504.000000,-15.549088,-344.000000,-351.968750,True,0,76561197989744167,apEX
3,"[Skeleton Knife, USP-S]",0,ropz,100.0,3.0,CT,2512.000000,144.999969,-504.000000,-343.367126,True,0,76561197991272318,ropz
4,"[Karambit, USP-S]",0,flameZ,100.0,3.0,CT,2584.000000,157.039948,-504.000000,-351.968750,True,0,76561197978835160,flameZ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1725105,"[Karambit, Flashbang]",19,jcobbb,77.0,3.0,CT,919.295776,154.196198,-1098.717407,-414.968750,True,172552,76561198178737429,jcobbb
1725106,[],19,mezii,0.0,2.0,TERRORIST,405.808990,56.407593,-912.780823,-391.468750,False,172552,76561197973140692,mezii
1725107,"[Butterfly Knife, USP-S, M4A1-S]",19,Twistzz,100.0,3.0,CT,1248.146973,147.721497,-1891.401733,-415.968750,True,172552,76561198016255205,Twistzz
1725108,"[Butterfly Knife, Flashbang]",19,broky,100.0,3.0,CT,757.513245,-129.346497,-398.611816,-415.968750,True,172552,76561198201620490,broky


In [ ]:
#20939 is when player spawns in for round next.

player[(player['tick'] > 114403) & (player['tick'] < 114412)]

#64 ticks after last start_round_beep we get our start of round

In [ ]:
np.where()

In [37]:
frames = {frame[0]:frame[1] for frame in test}

match_start = frames['round_announce_match_start']
round_starts = frames['cs_round_final_beep']
planted = frames['bomb_planted']
defused = frames['bomb_defused']
explode = frames['bomb_exploded']

In [49]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import re
import time

def get_hltv_demo_url(match_url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    }
    response = requests.get(match_url, headers=headers, impersonate="chrome")
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: Cloudflare still blocking.")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    demo_tag = soup.find('a', href=re.compile(r'/download/demo/\d+'))
    
    if demo_tag:
        return f"https://www.hltv.org{demo_tag['href']}"
    
    return "Demo link not found on page."

[ERROR] | Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xffff783daab0> 


In [50]:
get_hltv_demo_url('https://www.hltv.org/matches/2388130/vitality-vs-faze-starladder-budapest-major-2025')

Requesting: https://www.hltv.org/matches/2388130/vitality-vs-faze-starladder-budapest-major-2025


'https://www.hltv.org/download/demo/102981'

In [ ]:
def download_file(url, local_filename):
    with requests.get(url, stream=True, impersonate="chrome") as r:
        r.raise_for_status()
        with open(local_filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    return local_filename

In [ ]:
import subprocess
import os
import glob

def extract_with_unar(rar_path, extract_to="./demos"):
    os.makedirs(extract_to, exist_ok=True)
    
    try:
        result = subprocess.run(
            ['unar', '-f', '-o', extract_to, '-q', rar_path],
            check=True,
            capture_output=True,
            text=True
        )
        
        os.remove(rar_path)
        
        extracted_files = glob.glob(os.path.join(extract_to, "*.dem"))
        return extracted_files

    except subprocess.CalledProcessError as e:
        print(f"❌ unar failed: {e.stderr}")
        return []

In [52]:
#HLTV results at:
'https://www.hltv.org/results'

#Offset by n matches
'https://www.hltv.org/results?offset=200'
'offset=n'

#In order to get matches 600->800 we start at offset 600 and then also do 700
#as they go 100 at a time

#Min k star rating
'https://www.hltv.org/results?stars=1'
'stars=k'

'stars=k'

In [ ]:
#In results we have:
'https://www.hltv.org/matches/2388073/fnatic-vs-b8-starladder-budapest-major-2025-stage-2'

#In results we can look for 
r'matches/(\d+)/([^/\s]+)'

#using findall

In [60]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import re
import time

HEADERS = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    }

def scrape_results_page(results_url, offset):
    """
    Used in scrape_results() to get matches from each results page

    Args:
        results_url (str): The url to scrape for matches
        offset (int): The offset used in the url to get a certain page

    Returns:
        list: All urls scraped from this webpage
    """
    response = requests.get(results_url, headers=HEADERS, impersonate="chrome")
    
    if response.status_code != 200:
        print(f'Unable to reach results {offset} to {offset+100}')
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    matches = soup.find_all('a', href=re.compile(r'^/?matches/(\d+)/([^/\s]+)'))

    return [match['href'] for match in matches]

def scrape_results(min_star=1, min_offset=0, max_offset=1):
    """
    Using scrape_results_page() scrapes pages given a min_star rating and min and max offset

    Args:
        min_star (int): Minimum star rating, 1-5
        min_offset (int): Minimum offset. Show in 100s, so each number is 1 page
        max_offset (int): Maximum offset. 

    Returns:
        list: All urls of matches scraped
    """
    scraped_matches = []
    if max_offset <= min_offset:
        raise ValueError

    for num in range(min_offset, (max_offset)):
        offset = num * 100
        results_url = f'https://www.hltv.org/results?stars={min_star}&offset={offset}'

        temp_matches = scrape_results_page(results_url=results_url, offset=offset)
        if temp_matches:
            scraped_matches.extend(temp_matches)

        if num != max_offset-1:
            time.sleep(20)

    print(f'Scraped {len(scraped_matches)} from hltv.org')
    return scraped_matches

In [61]:
test = scrape_results()

Scraped 100 from hltv.org


In [63]:
test

['/matches/2388737/saw-vs-ninjas-in-pyjamas-roman-imperium-cup-iii',
 '/matches/2388736/gentle-mates-vs-ninjas-in-pyjamas-roman-imperium-cup-iii',
 '/matches/2388735/saw-vs-sharks-roman-imperium-cup-iii',
 '/matches/2388733/saw-vs-impulse-gw-roman-imperium-cup-iii',
 '/matches/2388714/saw-vs-exsad-roman-imperium-cup-iii',
 '/matches/2388711/saw-vs-wolf-roman-imperium-cup-iii',
 '/matches/2388130/vitality-vs-faze-starladder-budapest-major-2025',
 '/matches/2388129/faze-vs-natus-vincere-starladder-budapest-major-2025',
 '/matches/2388128/spirit-vs-vitality-starladder-budapest-major-2025',
 '/matches/2388127/furia-vs-natus-vincere-starladder-budapest-major-2025',
 '/matches/2388126/mouz-vs-faze-starladder-budapest-major-2025',
 '/matches/2388124/vitality-vs-the-mongolz-starladder-budapest-major-2025',
 '/matches/2388125/spirit-vs-falcons-starladder-budapest-major-2025',
 '/matches/2387486/ecstatic-vs-saw-galaxy-battle-2025-phase-5',
 '/matches/2388123/g2-vs-falcons-starladder-budapest-maj

In [8]:
parser = DemoParser(files[0])

In [77]:
parser.parse_header()

{'fullpackets_version': '2',
 'allow_clientside_entities': 'true',
 'allow_clientside_particles': 'true',
 'addons': '',
 'server_name': 'StarLadder Major Budapest 2025 | A',
 'demo_version_name': 'valve_demo_2',
 'demo_file_stamp': 'PBDEMS2\x00',
 'demo_version_guid': '8e9d71ab-04a1-4c01-bb61-acfede27c046',
 'game_directory': '/home/cs2server/cs-gs1/serverfiles/game/csgo',
 'client_name': 'SourceTV Demo',
 'map_name': 'de_nuke',
 'patch_version': '14126'}

In [78]:
parser.parse_voice()

[]

In [79]:
parser.list_updated_fields()

['CCSGameRulesProxy.CCSGameRules.m_MatchDevice',
 'CCSGameRulesProxy.CCSGameRules.m_MinimapVerticalSectionHeights',
 'CCSGameRulesProxy.CCSGameRules.m_TeamRespawnWaveTimes',
 'CCSGameRulesProxy.CCSGameRules.m_arrProhibitedItemIndices',
 'CCSGameRulesProxy.CCSGameRules.m_arrTournamentActiveCasterAccounts',
 'CCSGameRulesProxy.CCSGameRules.m_bAnyHostageReached',
 'CCSGameRulesProxy.CCSGameRules.m_bBlockersPresent',
 'CCSGameRulesProxy.CCSGameRules.m_bBombDropped',
 'CCSGameRulesProxy.CCSGameRules.m_bBombPlanted',
 'CCSGameRulesProxy.CCSGameRules.m_bCTCantBuy',
 'CCSGameRulesProxy.CCSGameRules.m_bCTTimeOutActive',
 'CCSGameRulesProxy.CCSGameRules.m_bFreezePeriod',
 'CCSGameRulesProxy.CCSGameRules.m_bGamePaused',
 'CCSGameRulesProxy.CCSGameRules.m_bGameRestart',
 'CCSGameRulesProxy.CCSGameRules.m_bHasMatchStarted',
 'CCSGameRulesProxy.CCSGameRules.m_bIsDroppingItems',
 'CCSGameRulesProxy.CCSGameRules.m_bIsHltvActive',
 'CCSGameRulesProxy.CCSGameRules.m_bIsQuestEligible',
 'CCSGameRulesProx

In [80]:
parser.parse_grenades()

,grenade_type,grenade_entity_id,x,y,z,tick,steamid,name
0,CFlashbang,1019,NaN,NaN,NaN,110,76561197989430253,karrigan
1,CFlashbang,1019,NaN,NaN,NaN,111,76561197989430253,karrigan
2,CFlashbang,1019,NaN,NaN,NaN,112,76561197989430253,karrigan
3,CFlashbang,1019,NaN,NaN,NaN,113,76561197989430253,karrigan
4,CFlashbang,1019,NaN,NaN,NaN,114,76561197989430253,karrigan
...,...,...,...,...,...,...,...,...
2136609,CSmokeGrenadeProjectile,535,112.21875,-1190.40625,-413.96875,172552,76561197989430253,karrigan
2136610,CSmokeGrenadeProjectile,594,421.62500,-1292.34375,-413.96875,172552,76561197973140692,mezii
2136611,CFlashbang,648,NaN,NaN,NaN,172552,76561198178737429,jcobbb
2136612,CFlashbang,887,NaN,NaN,NaN,172552,76561198201620490,broky


In [81]:
parser.parse_player_info()

,steamid,name,team_number
0,76561197989430253,karrigan,3
1,76561198113666193,ZywOo,2
2,76561197989744167,apEX,2
3,76561197991272318,ropz,2
4,76561197978835160,flameZ,2
5,76561198178737429,jcobbb,3
6,76561197973140692,mezii,2
7,76561198068422762,frozen,3
8,76561198016255205,Twistzz,3
9,76561198201620490,broky,3


In [82]:
parser.parse_item_drops()

,account_id,def_index,dropreason,inventory,item_id,paint_index,paint_seed,paint_wear,custom_name


In [83]:
parser.parse_header()

{'map_name': 'de_nuke',
 'client_name': 'SourceTV Demo',
 'allow_clientside_particles': 'true',
 'server_name': 'StarLadder Major Budapest 2025 | A',
 'patch_version': '14126',
 'fullpackets_version': '2',
 'allow_clientside_entities': 'true',
 'demo_file_stamp': 'PBDEMS2\x00',
 'demo_version_guid': '8e9d71ab-04a1-4c01-bb61-acfede27c046',
 'game_directory': '/home/cs2server/cs-gs1/serverfiles/game/csgo',
 'addons': '',
 'demo_version_name': 'valve_demo_2'}

In [84]:
parser.list_game_events()

['round_announce_match_point',
 'cs_intermission',
 'hltv_versioninfo',
 'round_announce_match_start',
 'hegrenade_detonate',
 'begin_new_match',
 'smokegrenade_expired',
 'cs_round_final_beep',
 'inferno_startburn',
 'cs_win_panel_match',
 'round_freeze_end',
 'player_disconnect',
 'player_connect_full',
 'player_hurt',
 'player_spawn',
 'announce_phase_end',
 'grenade_thrown',
 'cs_pre_restart',
 'player_sound',
 'hltv_chase',
 'cs_round_start_beep',
 'player_ping_stop',
 'player_ping',
 'player_death',
 'round_time_warning',
 'bomb_defused',
 'vote_cast',
 'player_team',
 'player_connect',
 'switch_team',
 'hltv_fixed',
 'round_announce_last_round_half',
 'bomb_pickup',
 'inferno_expire',
 'bomb_dropped',
 'weapon_fire',
 'server_cvar',
 'bomb_planted',
 'bomb_exploded',
 'smokegrenade_detonate',
 'entity_killed',
 'flashbang_detonate',
 'player_activate',
 'item_pickup']

In [24]:
player, events = parse_demo(files[0])

In [14]:
frames = {frame[0]:frame[1] for frame in events}

match_start = frames['round_announce_match_start'].loc[0, 'tick']
round_starts = frames['cs_round_final_beep']
match_end = frames['cs_win_panel_match'].loc[0, 'tick']
planted = frames['bomb_planted']
defused = frames['bomb_defused']
explode = frames['bomb_exploded']

In [15]:
players = player.copy()

In [16]:
players = players[players['tick'] <= match_end]
players = players[players['tick'] >= match_start]

In [18]:
players.dtypes

inventory               object
total_rounds_played      int32
is_match_started          bool
player_name             object
health                 float64
team_num               float64
team_name               object
X                      float32
yaw                    float32
Y                      float32
Z                      float32
is_alive                  bool
active_weapon_name      object
tick                     int32
steamid                 uint64
name                    object
dtype: object

In [19]:
planted.dtypes

site             int32
tick             int32
user_name       object
user_steamid    object
dtype: object

In [12]:
df = parser.parse_events(EVENT_PROPS, player=["last_place_name"])


In [13]:
df

[('round_announce_match_start',
     tick
  0  1471),
 ('bomb_exploded',
     site    tick user_last_place_name user_name       user_steamid
  0   263   20619               Garage    jcobbb  76561198178737429
  1   263   30014            BombsiteA  karrigan  76561197989430253
  2   326   67507               Secret  karrigan  76561197989430253
  3   326   76598                 Ramp     broky  76561198201620490
  4   263  137272              Squeaky     ZywOo  76561198113666193
  5   326  159807          Observation     mezii  76561197973140692),
 ('round_freeze_end',
        tick
  0     1471
  1    11281
  2    22219
  3    31614
  4    38213
  5    50077
  6    59418
  7    69107
  8    80667
  9    86868
  10   93422
  11  101394
  12  114406
  13  122545
  14  127971
  15  143953
  16  151477
  17  163993
  18  166966),
 ('cs_win_panel_match',
       tick
  0  172468),
 ('bomb_planted',
      site    tick user_last_place_name user_name       user_steamid
  0    326    8532          

In [58]:
death = parser.parse_events(['player_death'], player=["last_place_name"])

In [15]:
df

[('player_death',
       assistedflash assister_last_place_name assister_name   assister_steamid  \
  0            False                     None          None               None   
  1            False                   Heaven          apEX  76561197989744167   
  2            False                     None          None               None   
  3            False                     None          None               None   
  4            False                     None          None               None   
  ..             ...                      ...           ...                ...   
  136          False                     None          None               None   
  137          False                     None          None               None   
  138          False                     None          None               None   
  139          False                     None          None               None   
  140          False                     None          None               None  

In [59]:
death = death[0][1]

In [60]:
cols = ['attacker_steamid', 'attackerblind', 'attackerinair', 'assistedflash', 'assister_steamid', 'dmg_health', 'headshot', 'hitgroup', 'noscope', 'penetrated', 'thrusmoke', 'user_steamid', 'weapon', 'tick'] 

In [61]:
death = death[cols]

In [37]:
df = parser.parse_events(['grenade_thrown'], player=["last_place_name"])

In [43]:
df = df[0][1]

In [40]:
df2 = parser.parse_events(['player_hurt'], player=["last_place_name"])

In [44]:
df2 = df2[0][1]

In [68]:
df

,tick,user_last_place_name,user_name,user_steamid,weapon
0,2756,Outside,karrigan,76561197989430253,smokegrenade
1,2945,Roof,Twistzz,76561198016255205,molotov
2,3198,Outside,karrigan,76561197989430253,smokegrenade
3,3458,LockerRoom,apEX,76561197989744167,hegrenade
4,4350,Outside,karrigan,76561197989430253,flashbang
...,...,...,...,...,...
393,170271,Silo,flameZ,76561197978835160,smokegrenade
394,170364,Ramp,frozen,76561198068422762,incgrenade
395,170434,BombsiteA,jcobbb,76561198178737429,incgrenade
396,171101,Mini,karrigan,76561197989430253,smokegrenade


In [64]:
hurt = df2

In [62]:
death

,attacker_steamid,attackerblind,attackerinair,assistedflash,assister_steamid,dmg_health,headshot,hitgroup,noscope,penetrated,thrusmoke,user_steamid,weapon,tick
0,76561198068422762,False,False,False,None,11,False,chest,False,0,False,76561197978835160,glock,6252
1,76561198113666193,False,False,False,76561197989744167,114,True,head,False,0,False,76561197989430253,elite,6334
2,76561198016255205,False,False,False,None,11,False,chest,False,0,False,76561198113666193,glock,6557
3,76561197991272318,False,False,False,None,118,True,head,False,0,False,76561198068422762,usp_silencer,6688
4,76561197991272318,False,False,False,None,109,True,head,False,0,False,76561198178737429,usp_silencer,7482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,76561197989430253,False,False,False,None,24,False,chest,False,0,False,76561197991272318,m4a1_silencer,170232
137,76561198178737429,False,False,False,None,24,False,chest,False,0,False,76561197989744167,m4a1_silencer,170682
138,76561197989430253,False,False,False,None,24,False,chest,False,0,False,76561197978835160,m4a1_silencer,171953
139,76561198178737429,False,False,False,None,30,False,stomach,False,0,False,76561198113666193,m4a1_silencer,172153


In [75]:
hurt[hurt_cols]

,tick,user_steamid,health,dmg_health,weapon,attacker_steamid,hitgroup
0,3562,76561197989430253,50,50,hegrenade,76561197989744167,generic
1,3562,76561198178737429,96,4,hegrenade,76561197989744167,generic
2,4174,76561197989744167,85,14,glock,76561198201620490,chest
3,4212,76561197989744167,67,18,glock,76561198016255205,chest
4,5116,76561197989744167,46,20,,None,generic
...,...,...,...,...,...,...,...
481,172130,76561198178737429,77,22,tec9,76561198113666193,chest
482,172134,76561198113666193,43,24,m4a1,76561198178737429,chest
483,172146,76561198113666193,13,30,m4a1,76561198178737429,stomach
484,172153,76561198113666193,0,30,m4a1,76561198178737429,stomach


In [74]:
hurt_cols = [
    'tick',
    'user_steamid',
    'health',
    'dmg_health',
    'weapon',
    'attacker_steamid',
    'hitgroup',
]

In [69]:
hurt

,armor,attacker_last_place_name,attacker_name,attacker_steamid,dmg_armor,dmg_health,health,hitgroup,tick,user_last_place_name,user_name,user_steamid,weapon
0,0,Outside,apEX,76561197989744167,0,50,50,generic,3562,Outside,karrigan,76561197989430253,hegrenade
1,97,Outside,apEX,76561197989744167,3,4,96,generic,3562,Outside,jcobbb,76561198178737429,hegrenade
2,0,Outside,broky,76561198201620490,0,14,85,chest,4174,Outside,apEX,76561197989744167,glock
3,0,Silo,Twistzz,76561198016255205,0,18,67,chest,4212,Outside,apEX,76561197989744167,glock
4,0,None,None,None,0,20,46,generic,5116,Hell,apEX,76561197989744167,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,99,BombsiteA,ZywOo,76561198113666193,1,22,77,chest,172130,BombsiteA,jcobbb,76561198178737429,tec9
482,89,BombsiteA,jcobbb,76561198178737429,5,24,43,chest,172134,BombsiteA,ZywOo,76561198113666193,m4a1
483,83,BombsiteA,jcobbb,76561198178737429,6,30,13,stomach,172146,BombsiteA,ZywOo,76561198113666193,m4a1
484,77,BombsiteA,jcobbb,76561198178737429,6,30,0,stomach,172153,BombsiteA,ZywOo,76561198113666193,m4a1


In [ ]:
NADE_COLS = [
    'tick',
    'user_steamid',
    'weapon',
]